In [ ]:
import cv2
import tensorflow as tf
import sys
import numpy as np

sys.path.insert(
    0,
    "D:\KLASIFIKASI BERGANDA UNTUK OPTIMASI IMAGE DEBLURRING PADA CITRA DENGAN BLUR YANG BERVARIASI\library",
)
from utils import custom_range, preprocess_and_load_data, save_image, ssim

In [ ]:
import os
import random

with tf.device("/device:CPU:0"):
    dataset_path = "D:/Dataset/thumbnails128x128"
    file_names = os.listdir(dataset_path)

    total_samples = len(file_names)
    num_samples = 2000

    # Randomly select a subset of file names
    random.shuffle(file_names)
    selected_file_names = file_names[:num_samples]

    print(f"Number of samples: {total_samples}")
    print(f"Number of randomly selected samples: {len(selected_file_names)}")

In [ ]:
def determine_blur_level(ksize):
    if ksize in [3, 5, 7]:
        return "very_low"
    elif ksize in [9, 11, 13]:
        return "low"
    elif ksize in [15, 17, 19]:
        return "medium"
    elif ksize in [21, 23, 25]:
        return "high"
    elif ksize in [27, 29, 31]:
        return "very_high"
    else:
        return "unknown"

In [ ]:
def count_images_in_directory(directory):
    # Count the number of files in a directory (excluding subdirectories)
    return len(
        [
            name
            for name in os.listdir(directory)
            if os.path.isfile(os.path.join(directory, name))
        ]
    )

#### **GAUSSIAN BLUR SPLITTER**

In [ ]:
def add_gaussian_blur_splitter(
    images, range_set, output_dir, max_images_per_level=10000
):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    x_dir = os.path.join(output_dir)
    y_dir = os.path.join(output_dir)

    # Initialize dictionary to store image indices for each level
    level_img_indices = {}

    # Generate a custom range of odd numbers
    odd_numbers = custom_range(range_set[0], range_set[1], range_set[2])

    num_images = len(images)
    total_odd_numbers = len(odd_numbers)

    img_idx = 1  # Initialize image index

    # Iterate over each input image
    for i in range(num_images):
        odd_number = odd_numbers[i % total_odd_numbers]
        ksize = (odd_number, odd_number)

        # Apply Gaussian blur to the current image
        blurred_image = cv2.GaussianBlur(images[i], ksize, 0)
        level = determine_blur_level(odd_number)
        if level == "unknown":
            continue  # Skip saving the image for this level
        # Create directories for the current level if they don't exist
        level_x_dir = os.path.join(x_dir, "X", level)
        level_y_dir = os.path.join(y_dir, "Y", level)
        os.makedirs(level_x_dir, exist_ok=True)
        os.makedirs(level_y_dir, exist_ok=True)

        # Check the number of existing images in the current level directory
        current_level_image_count = len(os.listdir(level_x_dir))

        # Check if the maximum image count for the current level has been reached
        if current_level_image_count >= max_images_per_level:
            continue  # Skip saving the image for this level

        # Check if the level already has an index, if not initialize it
        if level not in level_img_indices:
            level_img_indices[level] = 1
        else:
            level_img_indices[level] += 1
        img_idx = level_img_indices[level]  # Get the current index for this level

        # Determine image paths for X (blurred) and Y (original) categories
        image_name = f"blurry_image_{img_idx}.png"
        x_image_path = os.path.join(level_x_dir, image_name)
        y_image_path = os.path.join(level_y_dir, image_name)

        # Save the blurred image and its corresponding original image
        save_image(blurred_image, x_image_path)
        save_image(images[i], y_image_path)
        img_idx += 1
        print(image_name)

#### **AVERAGE BLUR SPLITTER**

In [ ]:
def add_average_blur_splitter(
    images, range_set, output_dir, max_images_per_level=10000
):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    x_dir = os.path.join(output_dir)
    y_dir = os.path.join(output_dir)

    # Initialize dictionary to store image indices for each level
    level_img_indices = {}

    # Generate a custom range of odd numbers
    odd_numbers = custom_range(range_set[0], range_set[1], range_set[2])

    num_images = len(images)
    total_odd_numbers = len(odd_numbers)

    img_idx = 1  # Initialize image index

    # Iterate over each input image
    for i in range(num_images):
        odd_number = odd_numbers[i % total_odd_numbers]
        ksize = (odd_number, odd_number)

        # Apply Gaussian blur to the current image
        blurred_image = cv2.blur(images[i], ksize, 0)
        level = determine_blur_level(odd_number)
        if level == "unknown":
            continue  # Skip saving the image for this level
        # Create directories for the current level if they don't exist
        level_x_dir = os.path.join(x_dir, "X", level)
        level_y_dir = os.path.join(y_dir, "Y", level)
        os.makedirs(level_x_dir, exist_ok=True)
        os.makedirs(level_y_dir, exist_ok=True)

        # Check the number of existing images in the current level directory
        current_level_image_count = len(os.listdir(level_x_dir))

        # Check if the maximum image count for the current level has been reached
        if current_level_image_count >= max_images_per_level:
            continue  # Skip saving the image for this level

        # Check if the level already has an index, if not initialize it
        if level not in level_img_indices:
            level_img_indices[level] = 1
        else:
            level_img_indices[level] += 1
        img_idx = level_img_indices[level]  # Get the current index for this level

        # Determine image paths for X (blurred) and Y (original) categories
        image_name = f"blurry_image_{img_idx}.png"
        x_image_path = os.path.join(level_x_dir, image_name)
        y_image_path = os.path.join(level_y_dir, image_name)

        # Save the blurred image and its corresponding original image
        save_image(blurred_image, x_image_path)
        save_image(images[i], y_image_path)
        img_idx += 1
        print(image_name)

#### **MOTION BLUR SPLITTER**

In [ ]:
def add_motion_blur_splitter(images, range_set, output_dir, max_images_per_level=10000):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    x_dir = os.path.join(output_dir)
    y_dir = os.path.join(output_dir)

    # Initialize dictionary to store image indices for each level
    level_img_indices = {}

    # Generate a custom range of odd numbers
    odd_numbers = custom_range(range_set[0], range_set[1], range_set[2])

    num_images = len(images)
    total_odd_numbers = len(odd_numbers)

    img_idx = 1  # Initialize image index

    # Iterate over each input image
    for i in range(num_images):
        ksize = odd_numbers[i % total_odd_numbers]
        angle_of_motion = random.uniform(0, 360)
        # Apply Gaussian blur to the current image
        k = np.zeros((ksize, ksize), dtype=np.float32)
        k[(ksize - 1) // 2, :] = np.ones(ksize, dtype=np.float32)
        k = cv2.warpAffine(
            k,
            cv2.getRotationMatrix2D((ksize / 2 - 0.5, ksize / 2 - 0.5), angle_of_motion, 1.0),
            (ksize, ksize),
        )
        k = k * (1.0 / np.sum(k))
        blurred_image = cv2.filter2D(images[i], -1, k)
        level = determine_blur_level(ksize)
        if level == "unknown":
            continue  # Skip saving the image for this level
        # Create directories for the current level if they don't exist
        level_x_dir = os.path.join(x_dir, "X", level)
        level_y_dir = os.path.join(y_dir, "Y", level)
        os.makedirs(level_x_dir, exist_ok=True)
        os.makedirs(level_y_dir, exist_ok=True)
        # Check the number of existing images in the current level directory
        current_level_image_count = len(os.listdir(level_x_dir))

        # Check if the maximum image count for the current level has been reached
        if current_level_image_count >= max_images_per_level:
            continue  # Skip saving the image for this level

        # Check if the level already has an index, if not initialize it
        if level not in level_img_indices:
            level_img_indices[level] = 1
        else:
            level_img_indices[level] += 1
        img_idx = level_img_indices[level]  # Get the current index for this level

        # Determine image paths for X (blurred) and Y (original) categories
        image_name = f"blurry_image_{img_idx}.png"
        x_image_path = os.path.join(level_x_dir, image_name)
        y_image_path = os.path.join(level_y_dir, image_name)

        # Save the blurred image and its corresponding original image
        save_image(blurred_image, x_image_path)
        save_image(images[i], y_image_path)
        img_idx += 1
        print(image_name)

#### **SPLITTER**

In [ ]:
with tf.device("/device:GPU:0"):
    images = preprocess_and_load_data(dataset_path, selected_file_names)

In [ ]:
# Define the directory to save blurred images
output_dir = r"D:\KLASIFIKASI BERGANDA UNTUK OPTIMASI IMAGE DEBLURRING PADA CITRA DENGAN BLUR YANG BERVARIASI\dataset\For Deblurring\3x3\test"

In [ ]:
with tf.device("/device:GPU:0"):
    blurry_images = add_gaussian_blur_splitter(images, (3, 3, 2), output_dir)

In [ ]:
# import os
# import time

# # ... your training code ...

# # After your training is done, shut down the computer
# os.system("shutdown /s /t 1")